# ECB Speeches → Supervised Learning (TF-IDF)
This notebook assumes your **texts are speeches** (not press releases). It builds labels in two ways:

1. **Predict-the-next-change (forecasting):** for each speech date *S*, look forward up to *H* days; if the next MRO change is a **hike** label `+1`, if a **cut** label `-1`, else `0` (no change).  
2. **Window around changes (contemporaneous):** assign speeches within a \[-B business days, +F business days] window around an effective change date to `±1` and others to `0`.

Then it trains a **3-class** TF-IDF + Logistic model (−1/0/+1).

In [6]:
# --- Setup
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.tseries.offsets import BDay

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Optional helpers (fallbacks provided)
try:
    from skfin.text import coefs_plot, show_text
    _HAS_SKFIN = True
except Exception:
    _HAS_SKFIN = False
    def coefs_plot(df_coef, title="Coefficients"):
        s = df_coef.squeeze()
        top = s.nlargest(12)
        bot = s.nsmallest(12)
        fig, ax = plt.subplots(figsize=(8, 6))
        pd.concat([bot, top]).sort_values().plot(kind="barh", ax=ax)
        ax.set_title(title)
        plt.tight_layout()

    def show_text(df_text, lexica=None, n=None):
        for idx, row in df_text.iterrows():
            print(f"=== {idx} ===")
            txt = str(row["text"])
            print(txt[:800] + ("..." if len(txt) > 800 else ""))
            if lexica:
                print("\nTop positive:", list(lexica.get("positive", []).index))
                print("Top negative:", list(lexica.get("negative", []).index))
            print()

# Paths: try absolute then fallback to ./data
ABS_DATA = Path(r"C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset")
REL_DATA = Path("./data")
DATA_DIR = ABS_DATA if ABS_DATA.exists() else REL_DATA

ECB_SPEECHES_CSV     = DATA_DIR / "ecb_speeches_clean_minimal.csv"  # columns: date,text,(optional speaker/title/...)
ECB_POLICY_RATES_CSV = DATA_DIR / "ecb_policy_rates_daily_fake.csv" # replace with real daily/effective MRO file

print("[PATHS]")
print("DATA_DIR            :", DATA_DIR)
print("SPEECHES exists     :", ECB_SPEECHES_CSV.exists())
print("MRO daily exists    :", ECB_POLICY_RATES_CSV.exists())

[PATHS]
DATA_DIR            : C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset
SPEECHES exists     : True
MRO daily exists    : True


In [7]:
# --- Robust loaders & utilities
import csv

def _normalize_index_to_date_index(df, date_col="date"):
    if date_col in df.columns:
        s = pd.to_datetime(df[date_col], errors="coerce")
    else:
        s = pd.to_datetime(df.index, errors="coerce")
    s = s.dt.tz_localize(None).dt.normalize()
    df = df.copy()
    df.index = s
    df = df[~df.index.isna()].sort_index()
    df = df[~df.index.duplicated(keep="first")]
    return df

def _sniff_csv_params(path, sample_bytes=65536):
    with open(path, "rb") as f:
        raw = f.read(sample_bytes)
    enc = "utf-8-sig" if raw.startswith(b"\xef\xbb\xbf") else "utf-8"
    try:
        text = raw.decode(enc, errors="replace")
        dialect = csv.Sniffer().sniff(text, delimiters=[",",";","\t","|"])
        sep = dialect.delimiter
        quotechar = dialect.quotechar if dialect.quotechar else '"'
    except Exception:
        sep, quotechar = None, '"'
    return enc, sep, quotechar

def load_speeches(path=ECB_SPEECHES_CSV):
    print(f"\n[LOAD] ECB speeches from: {path}")
    enc, sep, quotechar = _sniff_csv_params(path)
    print(f"[SNIFF] encoding={enc} | sep={'auto' if sep is None else repr(sep)} | quotechar={repr(quotechar)}")
    tries = [
        dict(encoding=enc, sep=sep, engine="python", quotechar=quotechar, escapechar="\\"),
        dict(encoding=enc, sep=",",  engine="python", quotechar='"',     escapechar="\\"),
        dict(encoding=enc, sep=";",  engine="python", quotechar='"',     escapechar="\\"),
        dict(encoding="latin-1", sep=sep, engine="python", quotechar=quotechar, escapechar="\\")
    ]
    df = None; last_err = None
    for i, kw in enumerate(tries, 1):
        try:
            df = pd.read_csv(path, **{k:v for k,v in kw.items() if v is not None})
            print(f"[READ OK] try#{i} -> shape={df.shape}")
            break
        except Exception as e:
            print(f"[READ FAIL] try#{i}: {e}")
            last_err = e
    if df is None:
        raise last_err
    if "date" not in df.columns or "text" not in df.columns:
        raise AssertionError("Speeches CSV must contain 'date' and 'text'.")
    df = _normalize_index_to_date_index(df, "date")
    df = df[["text"]]
    print(f"[OK] speeches: shape={df.shape} | span={df.index.min().date()}→{df.index.max().date()}")
    return df

def load_mro_daily(path=ECB_POLICY_RATES_CSV):
    print(f"\n[LOAD] ECB MRO daily from: {path}")
    df = pd.read_csv(path)
    assert "date" in df.columns and "mro_rate" in df.columns, "MRO CSV must have 'date' and 'mro_rate'"
    df["mro_rate"] = pd.to_numeric(df["mro_rate"], errors="coerce")
    df = df.dropna(subset=["mro_rate"])
    df = _normalize_index_to_date_index(df, "date")
    print(f"[OK] MRO daily: shape={df.shape} | unique rates={df['mro_rate'].nunique()} | span={df.index.min().date()}→{df.index.max().date()}")
    return df

def compute_change_events(mro_daily: pd.DataFrame):
    s = mro_daily["mro_rate"].astype(float).sort_index()
    delta = s.diff()
    ev_idx = delta[delta.fillna(0) != 0].index
    events = pd.DataFrame(index=ev_idx, data={"change": np.sign(delta.loc[ev_idx]).astype(int).values})
    print(f"[EVENTS] changes: total={len(events)} | hikes={(events['change']==1).sum()} | cuts={(events['change']==-1).sum()}")
    return events

In [8]:
# --- Label speeches by the NEXT effective change within a forward horizon
def label_speeches_by_next_change(speeches: pd.DataFrame,
                                  events: pd.DataFrame,
                                  horizon_days=30,
                                  include_no_change_class=True):
    assert isinstance(speeches.index, pd.DatetimeIndex)
    assert isinstance(events.index, pd.DatetimeIndex)
    left  = speeches.assign(_s=speeches.index).reset_index(drop=True)[["_s","text"]]
    right = events.assign(_e=events.index).reset_index(drop=True)[["_e","change"]]
    merged = pd.merge_asof(left.sort_values("_s"),
                           right.sort_values("_e"),
                           left_on="_s", right_on="_e",
                           direction="forward", tolerance=pd.Timedelta(f"{horizon_days}D"))
    if include_no_change_class:
        out = speeches.copy()
        out["change_3cls"] = 0
        hits = merged["change"].notna()
        if hits.any():
            out.loc[merged.loc[hits,"_s"].values, "change_3cls"] = merged.loc[hits,"change"].astype(int).values
        print(f"[NEXT-CHANGE 3-CLASS] total={len(out)} | counts={out['change_3cls'].value_counts().to_dict()} | horizon={horizon_days}D")
        return out
    else:
        hits = merged["change"].notna()
        if not hits.any():
            print("[NEXT-CHANGE BINARY] No speeches matched within horizon.")
            return speeches.iloc[0:0].copy()
        idx = merged.loc[hits, "_s"].values
        out = speeches.loc[idx].copy()
        out["change"] = merged.loc[hits, "change"].astype(int).values
        print(f"[NEXT-CHANGE BINARY] labeled={len(out)} / {len(speeches)} | horizon={horizon_days}D | balance={out['change'].value_counts().to_dict()}")
        return out

In [9]:
# --- Label speeches by proximity to change dates (window); others = 0
def label_speeches_by_window(speeches: pd.DataFrame,
                             events: pd.DataFrame,
                             back_bdays=5, fwd_bdays=2, prefer_past=True,
                             include_no_change_class=True):
    assert isinstance(speeches.index, pd.DatetimeIndex)
    assert isinstance(events.index, pd.DatetimeIndex)
    s_idx = pd.DatetimeIndex(speeches.index).sort_values()
    ev_idx = pd.DatetimeIndex(events.index).sort_values()

    chosen = {}
    for e in ev_idx:
        window = set([e])
        for k in range(1, back_bdays+1): window.add(e - BDay(k))
        for k in range(1, fwd_bdays+1):  window.add(e + BDay(k))
        candidates = s_idx.intersection(pd.DatetimeIndex(sorted(window)))
        if len(candidates) == 0:
            continue
        def bd_dist(s, e):
            if s == e: return 0
            k = 0; cur = s
            if s < e:
                while cur < e: cur += BDay(1); k += 1
            else:
                while cur > e: cur -= BDay(1); k += 1
            return k
        bestS, bestKey = None, None
        for s in candidates:
            dist = bd_dist(s, e)
            tie  = 1 if (prefer_past and s <= e) else 0
            key  = (dist, -tie)
            if bestKey is None or key < bestKey:
                bestKey, bestS = key, s
        change = int(events.loc[e, "change"])
        prev = chosen.get(bestS)
        if prev is None or bestKey < prev[:2]:
            chosen[bestS] = (bestKey[0], bestKey[1], change)

    if include_no_change_class:
        out = speeches.copy()
        out["change_3cls"] = 0
        if chosen:
            S_dates = sorted(chosen.keys())
            out.loc[S_dates, "change_3cls"] = [chosen[s][2] for s in S_dates]
        print(f"[WINDOW 3-CLASS] total={len(out)} | counts={out['change_3cls'].value_counts().to_dict()} | window=[-{back_bdays}BD, +{fwd_bdays}BD]")
        return out
    else:
        if not chosen:
            print("[WINDOW BINARY] No speeches matched any window.")
            return speeches.iloc[0:0].copy()
        S_dates = sorted(chosen.keys())
        out = speeches.loc[S_dates].copy()
        out["change"] = [chosen[s][2] for s in S_dates]
        print(f"[WINDOW BINARY] labeled={len(out)} / {len(speeches)} | balance={out['change'].value_counts().to_dict()} | window=[-{back_bdays}BD, +{fwd_bdays}BD]")
        return out

In [10]:
# --- 3-class TF-IDF + Logistic
def train_tfidf_logistic_three_class(df_3: pd.DataFrame):
    X, y = df_3["text"], df_3["change_3cls"]
    if y.nunique() < 2 or len(y) < 25:
        print("[3-CLASS MODEL] Not enough samples.")
        return None, None
    est = Pipeline(steps=[
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=12000,
                                  lowercase=True, strip_accents="unicode",
                                  stop_words="english", token_pattern=r"\b[a-zA-Z]{3,}\b")),
        ("clf", LogisticRegression(
            penalty="elasticnet", solver="saga", l1_ratio=0.5,
            class_weight="balanced", max_iter=6000, C=1.0, n_jobs=-1, random_state=0
        )),
    ])
    est.fit(X, y)
    print("[3-CLASS MODEL] Trained on", len(X), "speeches.")
    return est, None

In [11]:
# --- Run end-to-end for speeches
speeches   = load_speeches()
mro_daily  = load_mro_daily()
events     = compute_change_events(mro_daily)

# A) Forecast next change from each speech (recommended)
df_3_next = label_speeches_by_next_change(speeches, events, horizon_days=30, include_no_change_class=True)
est3_next, _ = train_tfidf_logistic_three_class(df_3_next)

# B) Alternative: label by proximity to events (window)
df_3_win = label_speeches_by_window(speeches, events, back_bdays=5, fwd_bdays=2, prefer_past=True, include_no_change_class=True)
est3_win, _  = train_tfidf_logistic_three_class(df_3_win)


[LOAD] ECB speeches from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_speeches_clean_minimal.csv
[SNIFF] encoding=utf-8 | sep=',' | quotechar='"'
[READ OK] try#1 -> shape=(2939, 5)
[OK] speeches: shape=(2249, 1) | span=1997-02-07→2025-09-30

[LOAD] ECB MRO daily from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_policy_rates_daily_fake.csv
[OK] MRO daily: shape=(2827, 2) | unique rates=6 | span=2015-01-01→2025-10-31
[EVENTS] changes: total=11 | hikes=8 | cuts=3
[NEXT-CHANGE 3-CLASS] total=2249 | counts={0: 2179, 1: 43, -1: 27} | horizon=30D


ValueError: np.nan is an invalid document, expected byte or unicode string.

In [13]:
# =========================
# Evaluation & Diagnostics
# =========================
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

def time_split_eval(df_3, model=None, train_frac=0.8, max_features=12000):
    """
    Time-based split (no look-ahead): first train_frac for train, remainder for test.
    Prints classification report (macro-F1) and normalized confusion matrix.
    Returns the fitted model.
    """
    if df_3 is None or len(df_3) == 0 or "text" not in df_3.columns or ("change_3cls" not in df_3.columns):
        print("[TIME SPLIT] Provided dataframe is invalid. Expect columns: ['text','change_3cls'].")
        return None

    df_3 = df_3.sort_index()
    cut = int(len(df_3) * train_frac)
    train_idx = df_3.index[:cut]
    test_idx  = df_3.index[cut:]

    X_train, y_train = df_3.loc[train_idx, "text"], df_3.loc[train_idx, "change_3cls"]
    X_test,  y_test  = df_3.loc[test_idx,  "text"], df_3.loc[test_idx,  "change_3cls"]

    if model is None:
        model = Pipeline(steps=[
            ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=max_features,
                                      lowercase=True, strip_accents="unicode",
                                      stop_words="english", token_pattern=r"\b[a-zA-Z]{3,}\b")),
            ("clf", LogisticRegression(
                penalty="elasticnet", solver="saga", l1_ratio=0.5,
                class_weight="balanced", max_iter=6000, C=1.0,
                n_jobs=-1, random_state=0
            )),
        ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"[TIME SPLIT] sizes: {len(X_train)} train | {len(X_test)} test")
    print("\nClassification report (macro-F1 is key due to imbalance):")
    print(classification_report(y_test, y_pred, digits=3))

    cm = confusion_matrix(y_test, y_pred, labels=[-1, 0, 1], normalize="true")
    cm_df = pd.DataFrame(cm,
                         index=["true -1","true 0","true +1"],
                         columns=["pred -1","pred 0","pred +1"])
    print("\nNormalized confusion matrix (rows=true):")
    display(cm_df.round(3))
    return model

def top_tokens_per_class(trained_pipeline, topk=15):
    """
    Shows top positive/negative TF-IDF logistic weights per class.
    Works with a Pipeline(tfidf, clf) using LogisticRegression multiclass.
    """
    if trained_pipeline is None:
        print("[TOKENS] No trained pipeline provided.")
        return
    tfidf = trained_pipeline.named_steps.get("tfidf", None)
    clf   = trained_pipeline.named_steps.get("clf", None)
    if tfidf is None or clf is None or not hasattr(clf, "coef_"):
        print("[TOKENS] Pipeline must have steps tfidf and clf (LogisticRegression).")
        return

    # Rebuild vocabulary array in index order
    vocab_items = sorted(tfidf.vocabulary_.items(), key=lambda kv: kv[1])
    vocab = np.array([w for w, _ in vocab_items])  # shape (n_features,)
    classes = clf.classes_                          # e.g., [-1, 0, 1]
    coefs = clf.coef_                               # shape (n_classes, n_features)

    for cls, row in zip(classes, coefs):
        s = pd.Series(row, index=vocab).sort_values()
        print(f"\n=== Class {cls:+} — Bottom (most negative weights) ===")
        display(s.head(topk))
        print(f"=== Class {cls:+} — Top (most positive weights) ===")
        display(s.tail(topk))

# 1) Evaluate your "next-change" model using a clean time split
print("\n=== EVAL: Next-change labeling (30-day horizon) ===")
est3_next_eval = time_split_eval(df_3_next, model=est3_next, train_frac=0.8, max_features=12000)
top_tokens_per_class(est3_next_eval, topk=15)

# 2) Try a longer horizon to change label balance (e.g., 45 days) and re-evaluate
print("\n=== EVAL: Next-change labeling (45-day horizon) ===")
df_3_next_45 = label_speeches_by_next_change(speeches, events, horizon_days=45, include_no_change_class=True)
est3_next_45, _ = train_tfidf_logistic_three_class(df_3_next_45)
est3_next_45_eval = time_split_eval(df_3_next_45, model=est3_next_45, train_frac=0.8, max_features=12000)
top_tokens_per_class(est3_next_45_eval, topk=12)

# 3) Evaluate the window-based labeling too (if you ran it above)
print("\n=== EVAL: Window labeling (default −5BD, +2BD) ===")
if df_3_win is not None and "change_3cls" in df_3_win.columns and len(df_3_win) > 0:
    est3_win_eval = time_split_eval(df_3_win, model=est3_win, train_frac=0.8, max_features=12000)
    top_tokens_per_class(est3_win_eval, topk=12)
else:
    print("[EVAL] df_3_win is empty or missing 'change_3cls' — skip.")

# 4) (Optional) Restrict to President/Vice-President speeches if 'speaker' exists in your CSV
#    This can boost signal by focusing on the most market-relevant speeches.
try:
    raw = pd.read_csv(ECB_SPEECHES_CSV)
    if "speaker" in raw.columns:
        raw["date"] = pd.to_datetime(raw["date"], errors="coerce").dt.tz_localize(None).dt.normalize()
        sp = raw.dropna(subset=["date"]).sort_values("date")
        mask_pres = sp["speaker"].astype(str).str.contains(
            "Lagarde|Draghi|Trichet|Duisenberg|De Guindos", case=False, na=False
        )
        speeches_pres = sp.loc[mask_pres, ["date","text"]].dropna(subset=["text"])
        speeches_pres = speeches_pres.drop_duplicates("date").set_index("date").sort_index()
        print(f"\n[FILTER] president/VP speeches: {len(speeches_pres)}")

        if len(speeches_pres) >= 30:
            df_3_pres = label_speeches_by_next_change(speeches_pres, events, horizon_days=30, include_no_change_class=True)
            est3_pres, _ = train_tfidf_logistic_three_class(df_3_pres)
            print("=== EVAL: President/VP speeches (30-day horizon) ===")
            est3_pres_eval = time_split_eval(df_3_pres, model=est3_pres, train_frac=0.8, max_features=8000)
            top_tokens_per_class(est3_pres_eval, topk=12)
        else:
            print("[FILTER] Not enough president/VP speeches to train (need ~30+).")
    else:
        print("[FILTER] No 'speaker' column in speeches CSV — skipping speaker filter.")
except Exception as e:
    print("[FILTER] Speaker-based subset failed:", e)



=== EVAL: Next-change labeling (30-day horizon) ===


NameError: name 'est3_next' is not defined